![QuantConnect Logo](https://cdn.quantconnect.com/web/i/icon.png)
<hr>

## EODHD Upcoming IPOs Research

This notebook studies whether IPO offer price helps explain future returns

In [ ]:
qb = QuantBook()
# Anchor the research clock to the start of 2026 for a reproducible history window.
qb.set_start_date(2026, 1, 1)
# Daily bars will have an end_time that matches the following midnight.
qb.settings.daily_precise_end_time = False

### Build a Upcoming IPOs Universe

Select confirmed non-penny upcoming IPOs, then inspect the returned universe history.

In [ ]:
def select_assets(data: List[EODHDUpcomingIPOs]) -> List[Symbol]:
    # Keep expected/priced IPOs with a confirmed date and a >$1 minimum across the price band.
    return [d.symbol for d in data
            if d.ipo_date and d.deal_type in [EODHD.DealType.EXPECTED, EODHD.DealType.PRICED]
            and (prices := [x for x in [d.lowest_price, d.highest_price, d.offer_price] if x])
            and min(prices) > 1]

# Add the EODHD Upcoming IPOs universe.
universe = qb.add_universe(EODHDUpcomingIPOs, select_assets)
# Request universe history of the last 365 days.
universe_history = qb.universe_history(universe, qb.time - timedelta(365), qb.time - timedelta(1), flatten=True).rename_axis(['time', 'symbol']).drop(columns=['time'])
# Print the returned shape and columns.
print(f"Shape: {universe_history.shape}")
print(f"Columns: {list(universe_history.columns)}")
universe_history.head()

### Universe Diagnostics

Inspects the raw offer-price distribution and visualizes how the unique asset footprint expands chronologically.

In [ ]:
# Count selected assets by day.
universe_size = universe_history.reset_index().groupby(['time', 'symbol']).size().groupby('time').size()
print(f"Universe days: {len(universe_size)}")
# Store the selected symbol list.
unique_assets = list(universe_history.index.levels[1].unique())
print(f"Mean basket size per day: {universe_size.mean():.1f}")
print('')
print(universe_history.offerprice.describe().map('{:0.3f}'.format))
universe_size.plot(title='Daily Universe Size', ylabel='Universe Size');

### Daily Universe Prices

Fetch daily price history for every symbol that appears in the universe.

In [ ]:
# Extract unique assets
symbols = list(universe_history.index.get_level_values(1).unique())
# Fetch daily historical price metrics using the earliest timestamp available in the index.
history = qb.history(symbols, universe_history.index[0][0] - timedelta(1), qb.time, Resolution.DAILY)
history

### Align IPO Pricing And Returns

Build a joined table of IPO offer price and future returns.

In [ ]:
# Align the factor with the return from the next open to the following open.
dataset = (
    universe_history.reset_index().groupby(['time', 'symbol']).agg(offerprice=('offerprice', 'max'))
    .join(history.open.unstack('symbol').sort_index().pct_change(2, fill_method=None).shift(-2).stack().rename('futurereturn'), how='inner')
)

dataset.head()